# 📊 Matplotlib Tutorial 4: Complete EDA Project - Titanic Dataset

**Objective:** Apply all Matplotlib skills to perform comprehensive Exploratory Data Analysis

In this notebook, you'll:
- Perform complete EDA on the Titanic dataset
- Create publication-quality visualizations
- Extract meaningful insights
- Build a storytelling narrative with data

---

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

%matplotlib inline

# Set professional style
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 12
plt.rcParams['axes.titlesize'] = 16
plt.rcParams['axes.titleweight'] = 'bold'

## 1. Load and Explore the Dataset

In [ ]:
# Load data
df = pd.read_csv('../titanic.csv')

print("="*60)
print("TITANIC DATASET - EXPLORATORY DATA ANALYSIS")
print("="*60)
print(f"\nDataset Shape: {df.shape[0]} rows × {df.shape[1]} columns")
print(f"\nFirst 5 rows:")
df.head()

In [ ]:
# Dataset information
print("\n" + "="*60)
print("DATASET INFORMATION")
print("="*60)
df.info()

In [ ]:
# Statistical summary
print("\n" + "="*60)
print("STATISTICAL SUMMARY")
print("="*60)
df.describe()

In [ ]:
# Check missing values
missing = df.isnull().sum()
missing_pct = (missing / len(df)) * 100
missing_df = pd.DataFrame({'Missing Count': missing, 'Percentage': missing_pct})
missing_df = missing_df[missing_df['Missing Count'] > 0].sort_values('Missing Count', ascending=False)

print("\n" + "="*60)
print("MISSING VALUES")
print("="*60)
print(missing_df)

## 2. Univariate Analysis - Individual Variable Distributions

In [ ]:
# Create figure with subplots for key variables
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('Univariate Analysis - Key Variables', fontsize=20, fontweight='bold', y=0.98)

# Age distribution
axes[0, 0].hist(df['Age'].dropna(), bins=30, color='skyblue', edgecolor='black', alpha=0.7)
axes[0, 0].axvline(df['Age'].mean(), color='red', linestyle='--', linewidth=2, 
                   label=f'Mean: {df["Age"].mean():.1f}')
axes[0, 0].axvline(df['Age'].median(), color='green', linestyle=':', linewidth=2,
                   label=f'Median: {df["Age"].median():.1f}')
axes[0, 0].set_title('Age Distribution', fontsize=16)
axes[0, 0].set_xlabel('Age')
axes[0, 0].set_ylabel('Count')
axes[0, 0].legend()
axes[0, 0].grid(axis='y', alpha=0.3)

# Fare distribution
axes[0, 1].hist(df['Fare'], bins=30, color='lightgreen', edgecolor='black', alpha=0.7)
axes[0, 1].axvline(df['Fare'].mean(), color='red', linestyle='--', linewidth=2,
                   label=f'Mean: ${df["Fare"].mean():.2f}')
axes[0, 1].set_title('Fare Distribution', fontsize=16)
axes[0, 1].set_xlabel('Fare ($)')
axes[0, 1].set_ylabel('Count')
axes[0, 1].legend()
axes[0, 1].grid(axis='y', alpha=0.3)

# Passenger class
class_counts = df['Pclass'].value_counts().sort_index()
bars = axes[1, 0].bar(['1st Class', '2nd Class', '3rd Class'], class_counts.values,
                      color=['#FFD700', '#C0C0C0', '#CD7F32'], edgecolor='black')
for bar in bars:
    height = bar.get_height()
    axes[1, 0].text(bar.get_x() + bar.get_width()/2., height + 5,
                   f'{int(height)}', ha='center', va='bottom', fontweight='bold')
axes[1, 0].set_title('Passenger Class Distribution', fontsize=16)
axes[1, 0].set_ylabel('Count')
axes[1, 0].grid(axis='y', alpha=0.3)

# Survival
survival_counts = df['Survived'].value_counts()
colors = ['#FF6B6B', '#4ECDC4']
bars = axes[1, 1].bar(['Not Survived', 'Survived'], survival_counts.values,
                      color=colors, edgecolor='black')
for bar in bars:
    height = bar.get_height()
    pct = (height / len(df)) * 100
    axes[1, 1].text(bar.get_x() + bar.get_width()/2., height + 5,
                   f'{int(height)} ({pct:.1f}%)', ha='center', va='bottom', fontweight='bold')
axes[1, 1].set_title('Survival Distribution', fontsize=16)
axes[1, 1].set_ylabel('Count')
axes[1, 1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

**Key Insights:**
- Age is right-skewed with most passengers between 20-40 years
- Fare is highly right-skewed (most paid low fares, few paid very high)
- Majority of passengers were in 3rd class
- Only ~38% survived the disaster

## 3. Bivariate Analysis - Relationships Between Variables

In [ ]:
# Survival by Gender
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Bar chart
survival_by_sex = df.groupby('Sex')['Survived'].mean()
colors = ['#4169E1', '#FF69B4']
bars = axes[0].bar(survival_by_sex.index, survival_by_sex.values, 
                   color=colors, edgecolor='black', alpha=0.8)
for bar in bars:
    height = bar.get_height()
    axes[0].text(bar.get_x() + bar.get_width()/2., height + 0.02,
                f'{height:.2%}', ha='center', va='bottom', fontweight='bold', fontsize=14)
axes[0].set_title('Survival Rate by Gender', fontsize=16)
axes[0].set_ylabel('Survival Rate')
axes[0].set_ylim(0, 1)
axes[0].grid(axis='y', alpha=0.3)

# Pie charts for each gender
for idx, (sex, color) in enumerate(zip(['male', 'female'], ['#4169E1', '#FF69B4'])):
    sex_data = df[df['Sex'] == sex]['Survived'].value_counts()
    axes[1].pie([sex_data[0], sex_data[1]], 
                labels=[f'{sex.title()} - Died', f'{sex.title()} - Survived'],
                colors=['#FF6B6B', color], autopct='%1.1f%%',
                textprops={'fontsize': 11})

axes[1].set_title(f'Gender Survival Breakdown', fontsize=16)

plt.tight_layout()
plt.show()

In [ ]:
# Survival by Class and Gender
fig, ax = plt.subplots(figsize=(12, 8))

survival_by_class_sex = df.groupby(['Pclass', 'Sex'])['Survived'].mean().unstack()

x = np.arange(3)
width = 0.35

bars1 = ax.bar(x - width/2, survival_by_class_sex['female'], width,
               label='Female', color='#FF69B4', edgecolor='black', alpha=0.8)
bars2 = ax.bar(x + width/2, survival_by_class_sex['male'], width,
               label='Male', color='#4169E1', edgecolor='black', alpha=0.8)

# Add value labels
for bars in [bars1, bars2]:
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height + 0.02,
               f'{height:.2%}', ha='center', va='bottom', fontsize=11, fontweight='bold')

ax.set_xlabel('Passenger Class', fontsize=14)
ax.set_ylabel('Survival Rate', fontsize=14)
ax.set_title('Survival Rate by Class and Gender', fontsize=16)
ax.set_xticks(x)
ax.set_xticklabels(['1st Class', '2nd Class', '3rd Class'])
ax.legend(fontsize=12)
ax.grid(axis='y', alpha=0.3)
ax.set_ylim(0, 1.2)

plt.show()

**Key Insights:**
- Females had much higher survival rate (~74%) than males (~19%)
- Higher class passengers had better survival rates
- 1st class females had nearly 97% survival rate!
- "Women and children first" policy is evident

## 4. Age Analysis

In [ ]:
# Age distribution by survival
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Overlay histograms
axes[0].hist(df[df['Survived'] == 0]['Age'].dropna(), bins=30, alpha=0.6, 
             color='red', edgecolor='black', label='Not Survived')
axes[0].hist(df[df['Survived'] == 1]['Age'].dropna(), bins=30, alpha=0.6,
             color='green', edgecolor='black', label='Survived')
axes[0].set_title('Age Distribution by Survival', fontsize=16)
axes[0].set_xlabel('Age')
axes[0].set_ylabel('Count')
axes[0].legend()
axes[0].grid(axis='y', alpha=0.3)

# Box plot
age_by_survival = [df[df['Survived'] == 0]['Age'].dropna(), 
                   df[df['Survived'] == 1]['Age'].dropna()]
bp = axes[1].boxplot(age_by_survival, labels=['Not Survived', 'Survived'],
                     patch_artist=True)
colors = ['red', 'green']
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.6)
axes[1].set_title('Age by Survival - Box Plot', fontsize=16)
axes[1].set_ylabel('Age')
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Children survival analysis
df_copy = df.copy()
df_copy['IsChild'] = df_copy['Age'] < 18

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Children vs Adults survival
child_survival = df_copy.groupby('IsChild')['Survived'].mean()
bars = axes[0].bar(['Adults', 'Children'], child_survival.values,
                   color=['#4169E1', '#FFD700'], edgecolor='black', alpha=0.8)
for bar in bars:
    height = bar.get_height()
    axes[0].text(bar.get_x() + bar.get_width()/2., height + 0.02,
                f'{height:.2%}', ha='center', va='bottom', fontweight='bold', fontsize=14)
axes[0].set_title('Survival Rate: Adults vs Children', fontsize=16)
axes[0].set_ylabel('Survival Rate')
axes[0].set_ylim(0, 1)
axes[0].grid(axis='y', alpha=0.3)

# Age vs Fare scatter plot colored by survival
survived = df_copy[df_copy['Survived'] == 1]
not_survived = df_copy[df_copy['Survived'] == 0]

axes[1].scatter(not_survived['Age'], not_survived['Fare'], 
               alpha=0.5, color='red', label='Not Survived', s=50)
axes[1].scatter(survived['Age'], survived['Fare'], 
               alpha=0.5, color='green', label='Survived', s=50)
axes[1].axhline(y=0, color='black', linestyle='-', alpha=0.3)
axes[1].axvline(x=18, color='orange', linestyle='--', alpha=0.7, label='Age 18')
axes[1].set_title('Age vs Fare Colored by Survival', fontsize=16)
axes[1].set_xlabel('Age')
axes[1].set_ylabel('Fare ($)')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

**Key Insights:**
- Children had higher survival rates than adults
- Young children (0-10) had particularly high survival
- Age alone doesn't clearly separate survivors from non-survivors

## 5. Fare Analysis

In [ ]:
# Fare by class and survival
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Box plot - Fare by class
fare_by_class = [df[df['Pclass'] == c]['Fare'] for c in [1, 2, 3]]
bp = axes[0].boxplot(fare_by_class, labels=['1st Class', '2nd Class', '3rd Class'],
                     patch_artist=True)
colors = ['#FFD700', '#C0C0C0', '#CD7F32']
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
axes[0].set_title('Fare Distribution by Class', fontsize=16)
axes[0].set_ylabel('Fare ($)')
axes[0].grid(axis='y', alpha=0.3)

# Violin plot - Fare by survival
fare_by_survival = [df[df['Survived'] == 0]['Fare'], df[df['Survived'] == 1]['Fare']]
vp = axes[1].violinplot(fare_by_survival, positions=[0, 1], showmeans=True)
for pc, color in zip(vp['bodies'], ['red', 'green']):
    pc.set_facecolor(color)
    pc.set_edgecolor('black')
    pc.set_alpha(0.6)
axes[1].set_xticks([0, 1])
axes[1].set_xticklabels(['Not Survived', 'Survived'])
axes[1].set_title('Fare Distribution by Survival', fontsize=16)
axes[1].set_ylabel('Fare ($)')
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

**Key Insights:**
- 1st class passengers paid significantly higher fares
- Survivors tend to have paid higher fares (correlation with class)
- Many outliers in fare (very expensive tickets)

## 6. Correlation Analysis

In [ ]:
# Correlation heatmap
numeric_cols = ['Survived', 'Pclass', 'Age', 'SibSp', 'Parch', 'Fare']
corr_matrix = df[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(10, 8))

im = ax.imshow(corr_matrix.values, cmap='RdBu_r', aspect='auto', vmin=-1, vmax=1)

ax.set_xticks(range(len(numeric_cols)))
ax.set_yticks(range(len(numeric_cols)))
ax.set_xticklabels(numeric_cols, fontsize=12, rotation=45, ha='right')
ax.set_yticklabels(numeric_cols, fontsize=12)

# Add correlation values
for i in range(len(numeric_cols)):
    for j in range(len(numeric_cols)):
        val = corr_matrix.values[i, j]
        color = 'white' if abs(val) > 0.5 else 'black'
        ax.text(j, i, f'{val:.2f}', ha='center', va='center', 
               color=color, fontsize=11, fontweight='bold')

cbar = plt.colorbar(im, ax=ax)
cbar.set_label('Correlation Coefficient', fontsize=12)

ax.set_title('Correlation Matrix - Titanic Dataset', fontsize=16, pad=20)

plt.tight_layout()
plt.show()

**Key Correlations:**
- **Survived vs Pclass**: -0.34 (higher class = better survival)
- **Survived vs Fare**: 0.26 (higher fare = better survival)
- **Pclass vs Fare**: -0.55 (higher class = lower fare number but higher actual fare)
- **SibSp vs Parch**: 0.41 (family size correlation)

## 7. Family Size Analysis

In [ ]:
# Create family size feature
df_copy = df.copy()
df_copy['FamilySize'] = df_copy['SibSp'] + df_copy['Parch'] + 1  # +1 for the passenger
df_copy['IsAlone'] = (df_copy['FamilySize'] == 1).astype(int)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Family size distribution
family_counts = df_copy['FamilySize'].value_counts().sort_index()
axes[0].bar(family_counts.index, family_counts.values, 
           color='skyblue', edgecolor='black', alpha=0.7)
axes[0].set_title('Family Size Distribution', fontsize=16)
axes[0].set_xlabel('Family Size')
axes[0].set_ylabel('Count')
axes[0].grid(axis='y', alpha=0.3)

# Survival by family size
survival_by_family = df_copy.groupby('FamilySize')['Survived'].mean()
axes[1].plot(survival_by_family.index, survival_by_family.values, 
            color='green', linewidth=2, marker='o', markersize=8)
axes[1].axhline(y=df_copy['Survived'].mean(), color='red', linestyle='--',
               label=f'Overall: {df_copy["Survived"].mean():.2%}')
axes[1].set_title('Survival Rate by Family Size', fontsize=16)
axes[1].set_xlabel('Family Size')
axes[1].set_ylabel('Survival Rate')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

**Key Insights:**
- Most passengers traveled alone or in small families
- Small families (2-4 members) had better survival rates
- Very large families and solo travelers had lower survival

## 8. Comprehensive Summary Dashboard

In [ ]:
# Final comprehensive visualization
fig = plt.figure(figsize=(20, 12))
fig.suptitle('TITANIC DATASET - COMPREHENSIVE EDA DASHBOARD', 
             fontsize=24, fontweight='bold', y=0.98)

gs = fig.add_gridspec(3, 3, hspace=0.3, wspace=0.3)

# Row 1: Key distributions
ax1 = fig.add_subplot(gs[0, 0])
ax1.hist(df['Age'].dropna(), bins=30, color='skyblue', edgecolor='black', alpha=0.7)
ax1.set_title('Age Distribution', fontsize=14)
ax1.set_xlabel('Age')
ax1.grid(axis='y', alpha=0.3)

ax2 = fig.add_subplot(gs[0, 1])
ax2.hist(df['Fare'], bins=30, color='lightgreen', edgecolor='black', alpha=0.7)
ax2.set_title('Fare Distribution', fontsize=14)
ax2.set_xlabel('Fare ($)')
ax2.grid(axis='y', alpha=0.3)

ax3 = fig.add_subplot(gs[0, 2])
survival_counts = df['Survived'].value_counts()
ax3.pie(survival_counts.values, labels=['Died', 'Survived'], autopct='%1.1f%%',
        colors=['#FF6B6B', '#4ECDC4'], textprops={'fontsize': 11})
ax3.set_title('Overall Survival', fontsize=14)

# Row 2: Categorical analysis
ax4 = fig.add_subplot(gs[1, 0])
class_counts = df['Pclass'].value_counts().sort_index()
ax4.bar(['1st', '2nd', '3rd'], class_counts.values, 
       color=['#FFD700', '#C0C0C0', '#CD7F32'], edgecolor='black')
ax4.set_title('Passenger Class', fontsize=14)
ax4.set_ylabel('Count')
ax4.grid(axis='y', alpha=0.3)

ax5 = fig.add_subplot(gs[1, 1])
sex_counts = df['Sex'].value_counts()
ax5.bar(sex_counts.index, sex_counts.values, 
       color=['#4169E1', '#FF69B4'], edgecolor='black')
ax5.set_title('Gender Distribution', fontsize=14)
ax5.set_ylabel('Count')
ax5.grid(axis='y', alpha=0.3)

ax6 = fig.add_subplot(gs[1, 2])
embarked_counts = df['Embarked'].value_counts()
ax6.bar(embarked_counts.index, embarked_counts.values, 
       color=['#FF6B6B', '#4ECDC4', '#FFA07A'], edgecolor='black')
ax6.set_title('Embarkation Port', fontsize=14)
ax6.set_ylabel('Count')
ax6.grid(axis='y', alpha=0.3)

# Row 3: Key insights
ax7 = fig.add_subplot(gs[2, 0])
survival_by_sex = df.groupby('Sex')['Survived'].mean()
ax7.bar(survival_by_sex.index, survival_by_sex.values,
       color=['#4169E1', '#FF69B4'], edgecolor='black', alpha=0.8)
ax7.set_title('Survival by Gender', fontsize=14)
ax7.set_ylabel('Survival Rate')
ax7.set_ylim(0, 1)
ax7.grid(axis='y', alpha=0.3)

ax8 = fig.add_subplot(gs[2, 1])
survival_by_class = df.groupby('Pclass')['Survived'].mean()
ax8.bar(['1st', '2nd', '3rd'], survival_by_class.values,
       color=['#FFD700', '#C0C0C0', '#CD7F32'], edgecolor='black', alpha=0.8)
ax8.set_title('Survival by Class', fontsize=14)
ax8.set_ylabel('Survival Rate')
ax8.set_ylim(0, 1)
ax8.grid(axis='y', alpha=0.3)

ax9 = fig.add_subplot(gs[2, 2])
numeric_cols = ['Survived', 'Pclass', 'Age', 'SibSp', 'Parch', 'Fare']
corr_matrix = df[numeric_cols].corr()
im = ax9.imshow(corr_matrix.values, cmap='RdBu_r', aspect='auto', vmin=-1, vmax=1)
ax9.set_xticks(range(len(numeric_cols)))
ax9.set_yticks(range(len(numeric_cols)))
ax9.set_xticklabels(numeric_cols, fontsize=9, rotation=45, ha='right')
ax9.set_yticklabels(numeric_cols, fontsize=9)
for i in range(len(numeric_cols)):
    for j in range(len(numeric_cols)):
        val = corr_matrix.values[i, j]
        ax9.text(j, i, f'{val:.2f}', ha='center', va='center', 
                color='white' if abs(val) > 0.5 else 'black', fontsize=8)
ax9.set_title('Correlations', fontsize=14)
plt.colorbar(im, ax=ax9, fraction=0.046)

plt.tight_layout()
plt.savefig('titanic_eda_dashboard.png', dpi=300, bbox_inches='tight')
print("✓ Dashboard saved as 'titanic_eda_dashboard.png'")
plt.show()

## 9. Key Findings Summary

### 📊 Major Insights from Titanic EDA:

1. **Gender was the strongest predictor of survival**
   - Female survival rate: ~74%
   - Male survival rate: ~19%

2. **Class mattered significantly**
   - 1st class: ~63% survival
   - 2nd class: ~47% survival
   - 3rd class: ~24% survival

3. **Children had priority**
   - Children (< 18) had higher survival rates
   - Especially young children (0-10)

4. **Family size impact**
   - Small families (2-4) had better survival
   - Solo travelers and large families struggled

5. **Fare correlates with survival**
   - Higher fare passengers more likely to survive
   - Linked to passenger class

6. **Age distribution**
   - Most passengers: 20-40 years old
   - Wide age range: infants to 80 years

---

## 🎯 What You've Mastered:

✅ Complete EDA workflow  
✅ Multiple plot types (histograms, bar, pie, box, violin, scatter)  
✅ Subplots and custom layouts  
✅ Statistical visualizations  
✅ Annotations and customizations  
✅ Correlation analysis  
✅ Data storytelling  
✅ Professional-quality figures  

---

**Congratulations! You now have a strong command of Matplotlib! 🎉**